# Asmit EDA

In [197]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

## Data Integrity

Load data

In [198]:
sopp = pd.read_csv('./data/sopp_svi_merged.csv')
df = sopp.copy()
df.columns

Index(['Unnamed: 0', 'raw_row_number', 'date', 'time', 'service_area',
       'subject_age', 'subject_race', 'subject_sex', 'type', 'arrest_made',
       'citation_issued', 'warning_issued', 'outcome', 'contraband_found',
       'search_conducted', 'search_person', 'search_vehicle', 'search_basis',
       'reason_for_search', 'reason_for_stop', 'raw_action_taken',
       'raw_subject_race_description', 'serv', 'svi_rpl_themes'],
      dtype='str')

Basic data checks. Much of the missing data is in post-outcome features, which are not valid predictors of a search in any case because that information is not available before a stop is conducted.

In [199]:
print("=== BASIC OVERVIEW ===")
print("Shape:", df.shape)
print("Columns:", len(df.columns))

# Duplicates
print("\n=== DUPLICATES ===")
print("Exact duplicate rows:", int(df.duplicated().sum()))
for idcol in ["raw_row_number", "stop_id", "id"]:
    if idcol in df.columns:
        print(f"Duplicate values in {idcol}:", int(df[idcol].duplicated().sum()))

# Missingness (columns)
print("\n=== MISSINGNESS (TOP 10) ===")
miss = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=True)
}).sort_values(["pct_missing", "n_unique"], ascending=[False, False])

print(miss.head())

=== BASIC OVERVIEW ===
Shape: (383027, 24)
Columns: 24

=== DUPLICATES ===
Exact duplicate rows: 0
Duplicate values in raw_row_number: 0

=== MISSINGNESS (TOP 10) ===
                    dtype  n_missing  pct_missing  n_unique
reason_for_search     str     368749        96.27       632
search_basis          str     366739        95.75         5
contraband_found   object     366739        95.75         2
outcome               str      39172        10.23         3
arrest_made        object      34743         9.07         2


Check that categorical data is clean

In [200]:
# Categorical data
print("\n=== CATEGORICAL DATA CHECKS ===")

def cat_flags(s: pd.Series):
    """
    Checks if categorical column has has
    leading symbols or unnecessary whitespaces.
    """
    s = s.dropna().astype(str)
    if len(s) == 0:
        return dict(n_unique=0, pct_ws_edges=0.0, pct_leading_symbols=0.0, pct_double_spaces=0.0)

    return dict(
        n_unique=int(s.nunique()),
        pct_ws_edges=float((s.str.match(r"^\s+|\s+$").mean() * 100)),
        pct_leading_symbols=float((s.str.match(r"^[^\w]+").mean() * 100)),
        pct_double_spaces=float((s.str.contains(r"\s{2,}").mean() * 100)),
    )

cat_cols = [c for c in df.columns if df[c].dtype == "object"]
cat_rep = []
for c in cat_cols:
    rep = cat_flags(df[c])
    rep["col"] = c
    cat_rep.append(rep)

cat_rep = pd.DataFrame(cat_rep).sort_values(
    ["pct_leading_symbols", "pct_ws_edges", "pct_double_spaces", "n_unique"],
    ascending=False
)

print(cat_rep.head())


=== CATEGORICAL DATA CHECKS ===
   n_unique  pct_ws_edges  pct_leading_symbols  pct_double_spaces  \
0         2           0.0                  0.0                0.0   
1         2           0.0                  0.0                0.0   
2         2           0.0                  0.0                0.0   
3         2           0.0                  0.0                0.0   
4         2           0.0                  0.0                0.0   

                col  
0       arrest_made  
1   citation_issued  
2    warning_issued  
3  contraband_found  
4     search_person  


Missing data checks on numeric data

In [201]:
# Numeric data
print("\n=== NUMERIC DATA CHECKS===")
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and df[c].dtype != bool]
num_rep = []
for c in num_cols:
    x = df[c]
    num_rep.append({
        "col": c,
        "pct_missing": float(x.isna().mean() * 100),
        "min": float(np.nanmin(x)) if x.notna().any() else np.nan,
        "p1": float(np.nanpercentile(x.dropna(), 1)) if x.notna().any() else np.nan,
        "median": float(np.nanmedian(x)) if x.notna().any() else np.nan,
        "p99": float(np.nanpercentile(x.dropna(), 99)) if x.notna().any() else np.nan,
        "max": float(np.nanmax(x)) if x.notna().any() else np.nan,
        "n_unique": int(x.nunique(dropna=True))
    })

num_rep = pd.DataFrame(num_rep).sort_values(["pct_missing", "n_unique"], ascending=[False, False])
print("Shape of numeric subset:", num_rep.shape)
print(num_rep.head())


=== NUMERIC DATA CHECKS===
Shape of numeric subset: (4, 8)
              col  pct_missing         min           p1         median  \
1     subject_age     3.123279   10.000000    18.000000      34.000000   
2            serv     3.035556  110.000000   110.000000     510.000000   
3  svi_rpl_themes     3.035556    0.075382     0.075382       0.241848   
0      Unnamed: 0     0.000000    0.000000  3830.260000  191513.000000   

             p99            max  n_unique  
1      75.000000     100.000000        91  
2     930.000000     930.000000        19  
3       0.849382       0.849382        19  
0  379195.740000  383026.000000    383027  


Date/time parsing and some feature engineering

In [202]:
# Parse date
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year.astype("Int64")
df["month"] = df["date"].dt.month.astype("Int64")

# Day of week
df["day_of_week"] = df["date"].dt.day_name()

# Parse time
t1 = pd.to_datetime(df["time"], format="%H:%M:%S", errors="coerce")
t2 = pd.to_datetime(df["time"], format="%H:%M", errors="coerce")
time_parsed = t1.fillna(t2)

df["hour"] = time_parsed.dt.hour

# is_night: keep missing if hour missing
df["is_night"] = np.where(df["hour"].isna(), np.nan, (df["hour"] >= 18) | (df["hour"] < 6))
df["is_night"] = df["is_night"].astype("boolean")

Basic checks on the outcome `search_conducted`

In [203]:
print("Search conducted column checks")
print("dtype:", df["search_conducted"].dtype)
print("missing:", df["search_conducted"].isna().sum())

# Overall search rate
counts = df["search_conducted"].value_counts(dropna=False)
print("\nCounts:")
print(counts)

search_rate = df["search_conducted"].astype(int).mean()
print("Search percent:", round(100 * search_rate, 3), "%")

# Search rate + SVI by service area
tmp = pd.DataFrame({"service_area": df["service_area"].astype(str), "searched": y, "svi": df["svi_rpl_themes"]})
print("\nSearch rate by service_area (top 10 highest, min n=200):")
grp = tmp.groupby("service_area").agg(rate=("searched", "mean"), svi=("svi", "mean"), n=("searched", "size"))
grp = grp[grp["n"] >= 200].sort_values("rate", ascending=False)
print(grp.head(10))

Search conducted column checks
dtype: bool
missing: 0

Counts:
search_conducted
False    366739
True      16288
Name: count, dtype: int64
Search percent: 4.252 %

Search rate by service_area (top 10 highest, min n=200):
                  rate       svi      n
service_area                           
830           0.111170  0.757934  13295
440           0.107294  0.849382  13710
510           0.095388  0.549244  14897
820           0.090968  0.496553  12268
430           0.089688  0.653392  15632
810           0.054347  0.234884  13929
720           0.042398  0.683421  17430
620           0.041382  0.347316  20951
530           0.040062  0.321448    649
610           0.034949  0.121181  18999


Logical checks on `search_conducted` with related columns. We will drop these later, but they are useful to check that the column is labeled consistently.

In [204]:
# 1) Check that search_person/vehicle aligns with search_conducted
bad1 = df[(df["search_conducted"] == False) & ((df["search_person"] == True) | (df["search_vehicle"] == True))]
print("Check 1: searched person/vehicle but search_conducted==False")
print("Count:", len(bad1))
if len(bad1) > 0:
    print(bad1[["search_conducted","search_person","search_vehicle"]].head())

# 2) For searched stops, either search_person or search_vehicle
# should be True, if recorded
bad2 = df[(df["search_conducted"] == True) & (df["search_person"] == False) & (df["search_vehicle"] == False)]
print("Check 2: search_conducted==True but both search_person and search_vehicle are False")
print("Count:", len(bad2))
if len(bad2) > 0:
    print(bad2[["search_conducted","search_person","search_vehicle"]].head())

# 2.1) Extends 2) by checking when search_person/search_vehicle is missing
# for records in which a search was conducted
searched = df[df["search_conducted"] == True]
print("  Check 2.1: Among searched stops:")
print("  search_person missing %:", round(100 * searched["search_person"].isna().mean(), 2))
print("  search_vehicle missing %:", round(100 * searched["search_vehicle"].isna().mean(), 2))

# 2.2) Searched stops where both search_person and search_vehicle are missing
searched_both_missing = df[(df["search_conducted"] == True) & df["search_person"].isna() & df["search_vehicle"].isna()]
print("  Both search_person and search_vehicle missing:", round(100 * len(searched_both_missing) / len(searched), 2))


Check 1: searched person/vehicle but search_conducted==False
Count: 0
Check 2: search_conducted==True but both search_person and search_vehicle are False
Count: 0
  Check 2.1: Among searched stops:
  search_person missing %: 13.45
  search_vehicle missing %: 13.45
  Both search_person and search_vehicle missing: 13.45


## Feature Selection

Begin by removing features that contain post-stop information. These are naturally invalid predictors of search outcomes since they are recorded *after* the search is conducted or not conducted. Also drop unnecessary columns (e.g., `Unnamed: 0`) from the data wrangling step.

In [205]:
print("All Columns before dropping:")
for i, col in enumerate(df.columns.tolist()):
    print(col)

All Columns before dropping:
Unnamed: 0
raw_row_number
date
time
service_area
subject_age
subject_race
subject_sex
type
arrest_made
citation_issued
warning_issued
outcome
contraband_found
search_conducted
search_person
search_vehicle
search_basis
reason_for_search
reason_for_stop
raw_action_taken
raw_subject_race_description
serv
svi_rpl_themes
year
month
day_of_week
hour
is_night


Define columns to drop. We first drop obviously irrelevant columns. Most of these were created during the data wrangling stage. We also drop post-stop columns here 

In [206]:
# Drop post-stop information since it would cause data leakage
drop_post_stop = [
    "contraband_found",     # Recorded after search
    "search_basis",         # Recorded after search
    "reason_for_search",    # Recorded after search
    "search_person",        # Recorded after search
    "search_vehicle",       # Recorded after search
    "outcome",              # Downstream outcome
    "arrest_made",          # Downstream outcome
    "citation_issued",      # Downstream outcome
    "warning_issued",       # Downstream outcome
    "raw_action_taken"      # Downstream outcome
]

drop_irrelevant = [
    "Unnamed: 0",                   # Row id from data wrangling
    "raw_row_number",               # Row id from data wrangling
    "raw_subject_race_description", # Redundant with subject_race
    "date",                         # Redundant with engineered columns
    "time",                         # Redundant with engineered columns
    "serv",                         # Redundant with service_area
    "type",                         # No variance in column. Poor predictor.
]

Now we want to drop columns with a large amount of missing data. We noted earlier that missing data is fortunately not a big issue column-wise in the data set. Hence, we require that columns have at least 95% of their rows filled. From the below chunk, we can see that the only columns dropped are those which we would have dropped earlier anyway.

In [207]:
missing_pct = df.isna().mean().sort_values(ascending=False)
missing_thresh = 0.05 # Threshold (may want to tune)

drop_high_missing = missing_pct[missing_pct >= missing_thresh].index.tolist()
print("Columns with >= {:.0%} missing:".format(missing_thresh))
print(drop_high_missing)

if set(drop_high_missing).issubset(set(drop_post_stop + drop_irrelevant)):
    print("All of the above columns are a subset of the earlier dropped columns.")

Columns with >= 5% missing:
['reason_for_search', 'search_basis', 'contraband_found', 'outcome', 'arrest_made', 'citation_issued', 'raw_action_taken', 'warning_issued']
All of the above columns are a subset of the earlier dropped columns.


Now we will drop those columns.

In [208]:
drop_cols = list(set(drop_high_missing + drop_post_stop + drop_irrelevant))
print("\nDropping {} columns total:".format(len(drop_cols)))
df_dropped = df.drop(columns=drop_cols)

print("\nReduced df shape:", df_dropped.shape)
print("Remaining columns:")
print(df_dropped.columns)
print("\nPreview of the remaining data:")
print(df_dropped.head())


Dropping 17 columns total:

Reduced df shape: (383027, 12)
Remaining columns:
Index(['service_area', 'subject_age', 'subject_race', 'subject_sex',
       'search_conducted', 'reason_for_stop', 'svi_rpl_themes', 'year',
       'month', 'day_of_week', 'hour', 'is_night'],
      dtype='str')

Preview of the remaining data:
  service_area  subject_age            subject_race subject_sex  \
0          110         24.0                   white        male   
1          320         42.0                   white        male   
2          320         29.0  asian/pacific islander        male   
3          610         23.0                   white        male   
4          930         35.0                hispanic        male   

   search_conducted      reason_for_stop  svi_rpl_themes  year  month  \
0             False     Moving Violation        0.241848  2014      1   
1             False     Moving Violation        0.213643  2014      1   
2             False     Moving Violation        0.21364

# Data Pre-processing

In [209]:
df = df_dropped.copy()

# Helper to normalize string labels
def normalize_label(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = re.sub(r"^[^\w]+", "", x)  # remove unnecessary leading characters
    x = re.sub(r"\s+", " ", x)     # remove repeated whitespace
    return x

# Clean/standardize specific fields

# service_area: treat 'Unknown' as missing
if "service_area" in df.columns:
    df["service_area"] = df["service_area"].map(normalize_label)
    df.loc[df["service_area"].astype(str).str.lower().isin(["unknown", "nan", "none", ""]), "service_area"] = pd.NA

# year/month: convert floats to integers first
for c in ["year", "month"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

print(df.head())

  service_area  subject_age            subject_race subject_sex  \
0          110         24.0                   white        male   
1          320         42.0                   white        male   
2          320         29.0  asian/pacific islander        male   
3          610         23.0                   white        male   
4          930         35.0                hispanic        male   

   search_conducted      reason_for_stop  svi_rpl_themes  year  month  \
0             False     Moving Violation        0.241848  2014      1   
1             False     Moving Violation        0.213643  2014      1   
2             False     Moving Violation        0.213643  2014      1   
3             False     Moving Violation        0.121181  2014      1   
4             False  Equipment Violation        0.075382  2014      1   

  day_of_week  hour  is_night  
0   Wednesday   1.0      True  
1   Wednesday   5.0      True  
2   Wednesday   7.0     False  
3   Wednesday   8.0     False 

Encoding categorical variables. We will begin with the only two binary fields.

In [210]:
df["subject_sex"] = df["subject_sex"].map({"male": 1, "female": 0})  # NaN stays NaN
df["search_conducted"] = df["search_conducted"].astype(int)          # False->0, True->1

Now, we will convert categorical fields and observe their levels.

In [211]:
# Define categorical (> 2 levels) predictors
cat_cols = [
    "subject_race",
    "year",
    "month",
    "reason_for_stop",
    "service_area",
    "day_of_week",
]

cat_cols = [c for c in cat_cols if c in df.columns]

# Normalize labels for strings and convert to category
for c in cat_cols:
    if df[c].dtype == "object":
        df[c] = df[c].map(normalize_label)

for c in cat_cols:
    df[c] = df[c].astype("category")

print("Converted to category:", cat_cols)

# Inspect levels + counts
for c in cat_cols:
    print(f"\n=== {c} ===")
    print("dtype:", df[c].dtype)
    print("n_unique:", df[c].nunique(dropna=True))
    print("missing %:", round(df[c].isna().mean() * 100, 3))
    print("Top levels:")
    print(df[c].value_counts(dropna=False).head(15))

Converted to category: ['subject_race', 'year', 'month', 'reason_for_stop', 'service_area', 'day_of_week']

=== subject_race ===
dtype: category
n_unique: 5
missing %: 0.322
Top levels:
subject_race
white                     162226
hispanic                  117083
black                      42705
asian/pacific islander     32541
other                      27238
NaN                         1234
Name: count, dtype: int64

=== year ===
dtype: category
n_unique: 4
missing %: 0.048
Top levels:
year
2014    139359
2015    112736
2016    102607
2017     28142
NaN        183
Name: count, dtype: int64

=== month ===
dtype: category
n_unique: 12
missing %: 0.048
Top levels:
month
2      43349
3      42474
1      38847
4      33590
5      31409
6      30304
7      29115
8      28534
11     27801
10     26744
9      25528
12     25149
NaN      183
Name: count, dtype: int64

=== reason_for_stop ===
dtype: category
n_unique: 97
missing %: 0.057
Top levels:
reason_for_stop
Moving Violation           

As we can see, there are 97 observed reasons for stop. Creating 96 dummy variables may lead to a less powerful model fit. It seems that most of the occurances are one of 6 stops. So, we will classify all others as being in the category `Other`.

In [212]:
K = 6
r = df["reason_for_stop"].astype("object")
topk = r.value_counts(dropna=True).head(K).index.tolist()
print("Top K reasons for stop:", topk)

# Classify everything else as "Other"
df["reason_for_stop"] = np.where(
    r.isna(),
    np.nan,
    np.where(r.isin(topk), r, "Other")
)

# Convert back to category
df["reason_for_stop"] = df["reason_for_stop"].astype("category")
df["reason_for_stop"] = df["reason_for_stop"].cat.remove_unused_categories()

print("\nNew reason_for_stop counts:")
print(df["reason_for_stop"].value_counts(dropna=False))
print("\nNumber of levels:", df["reason_for_stop"].nunique(dropna=True))

Top K reasons for stop: ['Moving Violation', 'Equipment Violation', 'Radio Call/Citizen Contact', 'Muni, County, H&S Code', 'Personal Knowledge/Informant', 'Suspect Info (I.S., Bulletin, Log)']

New reason_for_stop counts:
reason_for_stop
Moving Violation                      279839
Equipment Violation                    97372
Radio Call/Citizen Contact              1886
Muni, County, H&S Code                  1291
Other                                   1034
Personal Knowledge/Informant             860
Suspect Info (I.S., Bulletin, Log)       526
NaN                                      219
Name: count, dtype: int64

Number of levels: 7


Next, we will handle missingness in the remaining data set. Above, we can see that most of the columns are full. If the amount of data present by removing records that contain *any* missing values is sufficient, we may proceed by only retaining complete cases. This appears to be the case.

In [213]:
model_cols = df.columns.tolist()

# Check 
complete_mask = df[model_cols].notna().all(axis=1)
print("\nTotal rows:", len(df))
print("Complete-case rows:", int(complete_mask.sum()))
print("Dropped rows:", int((~complete_mask).sum()))
print("Percent dropped:", round(100 * (~complete_mask).mean(), 2), "%")

# Which columns drive most row drops
missing_any = df[model_cols].isna()
drop_drivers = missing_any.mean().sort_values(ascending=False)
print("\nMissing rate by column:")
print((drop_drivers * 100).round(3))


Total rows: 383027
Complete-case rows: 358690
Dropped rows: 24337
Percent dropped: 6.35 %

Missing rate by column:
subject_age         3.123
svi_rpl_themes      3.036
service_area        2.935
subject_race        0.322
hour                0.192
is_night            0.192
subject_sex         0.173
reason_for_stop     0.057
year                0.048
month               0.048
day_of_week         0.048
search_conducted    0.000
dtype: float64


Make the complete-case data set

In [214]:
df_cc = df.dropna(subset=model_cols).copy()
# Remove unused categories
cat_cols = df_cc.select_dtypes(include="category").columns
for c in cat_cols:
    df_cc[c] = df_cc[c].cat.remove_unused_categories()
print("df_cc shape:", df_cc.shape)

df_cc shape: (358690, 12)


Here, we are making some final quality of life changes. We will rename the column containing the SVI information and change the data type for the subject sex column.

In [224]:
df_cc = df_cc.rename(columns={"svi_rpl_themes": "svi"})
df_cc["subject_sex"] = pd.Categorical(df_cc["subject_sex"]).codes.astype("int64")

## Encoding / Final selection

# Interactions / Correlation

olivia plot `search rate by race across svi` suggests there is a interaction between race and svi

gianluca says is_night is redundant and we should remove and keep hour of day

## Model Assumption Validation